# 開 LLM 伺服器

每次用之前在 server 電腦打開，由上而下按 ▶ 執行。第一次用要先跑完 `install_server.ipynb`。

## 1. 設定

In [ ]:
import ipaddress
import json
import os
import socket
import subprocess
import tempfile
import time
import urllib.request
from pathlib import Path

MODEL = "qwen3:14b"  # 想換模型改這裏；install_server 和 start_server 兩個檔案要一樣
OLLAMA = Path(os.environ["LOCALAPPDATA"]) / "Programs" / "Ollama" / "ollama.exe"
URL = "http://127.0.0.1:11434"
RULE = "Ollama LLM Server"
LOG = Path(tempfile.gettempdir()) / "ollama_server.log"


def ollama_up():
    try:
        urllib.request.urlopen(URL, timeout=2)
        return True
    except OSError:
        return False


def wait_up(seconds=60):
    for _ in range(seconds):
        if ollama_up():
            return True
        time.sleep(1)
    return False


def post(path, body, timeout=600):
    req = urllib.request.Request(URL + path, json.dumps(body).encode("utf-8"),
                                 {"Content-Type": "application/json"})
    return urllib.request.urlopen(req, timeout=timeout)


print("設定完成，模型：", MODEL)

## 2. 開伺服器

執行後會顯示 IP，然後一直運行。

- **要停止**：按這個 cell 左邊的停止鍵 ■
- 直接關掉 VS Code 的話伺服器**不會停**，要再開這個檔案執行 1 和 2，然後按 ■

In [ ]:
# 開機自動開的 Ollama 只聽本機，先關掉，再用下面的設定重開
for name in ("ollama app.exe", "ollama.exe"):
    subprocess.run(["taskkill", "/f", "/im", name], capture_output=True)
time.sleep(2)

env = {**os.environ,
       "OLLAMA_HOST": "0.0.0.0:11434",   # 讓其他電腦連得到
       "OLLAMA_KEEP_ALIVE": "-1",        # 模型一直留在顯存
       "OLLAMA_CONTEXT_LENGTH": "8192"}
log = open(LOG, "w", encoding="utf-8")
server = subprocess.Popen([str(OLLAMA), "serve"], env=env, stdout=log, stderr=subprocess.STDOUT,
                          creationflags=subprocess.CREATE_NO_WINDOW)
if not wait_up():
    server.kill()
    raise RuntimeError(f"Ollama 60 秒內沒有啟動，記錄在 {LOG}")

tailscale = ipaddress.ip_network("100.64.0.0/10")
for ip in sorted({a[4][0] for a in socket.getaddrinfo(socket.gethostname(), None, socket.AF_INET)}):
    if ipaddress.ip_address(ip) in tailscale:
        print(f"Tailscale IP：{ip}   ← 填到聊天電腦 config/llm.json 的 server.base_url")
    elif not ip.startswith("169.254."):
        print(f"區域網絡 IP：{ip}")

print("\n載入模型並測試…")
t = time.time()
with post("/v1/chat/completions", {
        "model": MODEL, "max_tokens": 100, "reasoning_effort": "none",
        "messages": [{"role": "user", "content": "用一句話打招呼"}]}) as r:
    reply = json.loads(r.read())["choices"][0]["message"]["content"]
print(f"測試回覆（{time.time() - t:.0f} 秒）：{reply.strip()}")

print("\n伺服器運行中。要停止：按這個 cell 左邊的 ■")
try:
    server.wait()
    print(f"伺服器自己停了，記錄在 {LOG}")
except KeyboardInterrupt:
    server.terminate()
    server.wait()
    print("伺服器已停止")
finally:
    log.close()